In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv(
    r"C:\Users\Usuario\Documents\mdc\TradingBacktest\datos\ES_1min.txt",
    sep=";",
    header=None,
    names=["datetime", "open", "high", "low", "close", "volume"]
)

df["datetime"] = pd.to_datetime(df["datetime"], format="%Y%m%d %H%M%S")
df = df.set_index("datetime")

print(f"Total de barras: {len(df)}")
print(f"Desde: {df.index[0]}")
print(f"Hasta: {df.index[-1]}")
print(df.head())

Total de barras: 35805
Desde: 2026-03-12 04:01:00
Hasta: 2026-04-17 12:55:00
                        open     high      low    close  volume
datetime                                                       
2026-03-12 04:01:00  6762.50  6762.50  6762.50  6762.50       1
2026-03-12 04:03:00  6764.00  6764.25  6763.75  6764.25       9
2026-03-12 04:05:00  6764.75  6766.00  6764.75  6765.75      24
2026-03-12 04:06:00  6766.00  6767.00  6766.00  6766.25      40
2026-03-12 04:07:00  6765.75  6766.00  6765.75  6766.00       3


In [3]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv(
    r"C:\Users\Usuario\Documents\mdc\TradingBacktest\datos\ES_1min.txt",
    sep=";",
    header=None,
    names=["datetime", "open", "high", "low", "close", "volume"]
)

df["datetime"] = pd.to_datetime(df["datetime"], format="%Y%m%d %H%M%S")
df = df.set_index("datetime")

print(f"Total de barras: {len(df)}")
print(f"Desde: {df.index[0]}")
print(f"Hasta: {df.index[-1]}")
print(df.head())

Total de barras: 35805
Desde: 2026-03-12 04:01:00
Hasta: 2026-04-17 12:55:00
                        open     high      low    close  volume
datetime                                                       
2026-03-12 04:01:00  6762.50  6762.50  6762.50  6762.50       1
2026-03-12 04:03:00  6764.00  6764.25  6763.75  6764.25       9
2026-03-12 04:05:00  6764.75  6766.00  6764.75  6765.75      24
2026-03-12 04:06:00  6766.00  6767.00  6766.00  6766.25      40
2026-03-12 04:07:00  6765.75  6766.00  6765.75  6766.00       3


In [5]:
import pytz

eastern = pytz.timezone("America/New_York")
df.index = df.index.tz_localize("America/New_York")

# Filtrar solo el período SHORT y horario de sesión regular
start = "2026-03-12"
end = "2026-04-01"
df_period = df[start:end]

# Filtrar horario 9:30 a 16:00 ET
df_session = df_period.between_time("09:30", "16:00")

# Construir velas de 5 minutos
df_5m = df_session.resample("5min").agg({
    "open": "first",
    "high": "max",
    "low": "min",
    "close": "last",
    "volume": "sum"
}).dropna()

print(f"Total barras de 5 minutos: {len(df_5m)}")
print(f"Desde: {df_5m.index[0]}")
print(f"Hasta: {df_5m.index[-1]}")
print(df_5m.head(10))

Total barras de 5 minutos: 1185
Desde: 2026-03-12 09:30:00-04:00
Hasta: 2026-04-01 16:00:00-04:00
                              open     high      low    close  volume
datetime                                                             
2026-03-12 09:30:00-04:00  6792.50  6797.00  6791.50  6796.50     117
2026-03-12 09:35:00-04:00  6795.75  6799.00  6795.00  6798.00      71
2026-03-12 09:40:00-04:00  6799.00  6805.00  6798.50  6804.00     217
2026-03-12 09:45:00-04:00  6804.25  6807.00  6803.00  6805.75     118
2026-03-12 09:50:00-04:00  6805.50  6808.50  6803.50  6808.50      83
2026-03-12 09:55:00-04:00  6808.75  6809.00  6806.50  6807.25      82
2026-03-12 10:00:00-04:00  6807.00  6810.25  6806.50  6807.50     124
2026-03-12 10:05:00-04:00  6807.75  6808.50  6804.50  6804.75      89
2026-03-12 10:10:00-04:00  6804.25  6804.75  6802.50  6803.25      36
2026-03-12 10:15:00-04:00  6803.75  6806.25  6801.75  6803.25      64


In [6]:
trades = []
dias = df_5m.groupby(df_5m.index.date)

for fecha, dia in dias:
    barras = dia.copy()
    if len(barras) < 6:
        continue
    
    # Rango ORB primeras 3 velas (9:30, 9:35, 9:40)
    orb = barras.iloc[:3]
    orb_high = orb["high"].max()
    orb_low = orb["low"].min()
    orb_rango = orb_high - orb_low
    
    # Filtro volatilidad
    if orb_rango > 20:
        continue
    
    # Buscar entrada SHORT desde vela 4 en adelante
    entrada = None
    for i in range(3, len(barras)):
        vela = barras.iloc[i]
        if vela["close"] < orb_low:
            entrada = {
                "fecha": fecha,
                "hora_entrada": barras.index[i],
                "precio_entrada": vela["close"],
                "orb_high": orb_high,
                "orb_low": orb_low,
                "orb_rango": orb_rango,
                "stop": orb_high,
                "target": vela["close"] - (orb_rango * 1.5)
            }
            break
    
    if entrada is None:
        continue
    
    # Simular resultado
    resultado = None
    for i in range(barras.index.get_loc(entrada["hora_entrada"]) + 1, len(barras)):
        vela = barras.iloc[i]
        hora = barras.index[i].time()
        
        # Cierre forzado 15:45
        if hora >= pd.Timestamp("15:45").time():
            entrada["salida"] = vela["close"]
            entrada["hora_salida"] = barras.index[i]
            entrada["resultado"] = "tiempo"
            entrada["pnl"] = entrada["precio_entrada"] - vela["close"]
            resultado = entrada
            break
        
        # Stop hit
        if vela["high"] >= entrada["stop"]:
            entrada["salida"] = entrada["stop"]
            entrada["hora_salida"] = barras.index[i]
            entrada["resultado"] = "stop"
            entrada["pnl"] = entrada["precio_entrada"] - entrada["stop"]
            resultado = entrada
            break
        
        # Target hit
        if vela["low"] <= entrada["target"]:
            entrada["salida"] = entrada["target"]
            entrada["hora_salida"] = barras.index[i]
            entrada["resultado"] = "target"
            entrada["pnl"] = entrada["precio_entrada"] - entrada["target"]
            resultado = entrada
            break
    
    if resultado:
        trades.append(resultado)

df_trades = pd.DataFrame(trades)
print(f"Total trades: {len(df_trades)}")
print(df_trades[["fecha","precio_entrada","stop","target","resultado","pnl"]].to_string())

Total trades: 13
         fecha  precio_entrada     stop    target resultado     pnl
0   2026-03-12         6788.75  6805.00  6768.500    target  20.250
1   2026-03-13         6716.25  6726.25  6708.375      stop -10.000
2   2026-03-17         6735.50  6748.00  6725.375      stop -12.500
3   2026-03-18         6806.25  6810.00  6801.375      stop  -3.750
4   2026-03-19         6645.50  6663.25  6619.625    target  25.875
5   2026-03-20         6638.00  6654.50  6618.125    target  19.875
6   2026-03-23         6498.25  6518.75  6468.250      stop -20.500
7   2026-03-24         6607.25  6626.25  6590.000      stop -19.000
8   2026-03-25         6651.75  6662.50  6642.375      stop -10.750
9   2026-03-26         6577.00  6591.75  6561.250      stop -14.750
10  2026-03-27         6518.25  6532.00  6504.000    target  14.250
11  2026-03-30         6430.50  6440.50  6417.750    target  12.750
12  2026-03-31         6427.75  6446.25  6414.250      stop -18.500


In [7]:
# Valor del punto ES = $50 por punto
punto = 50

df_trades["pnl_usd"] = df_trades["pnl"] * punto
df_trades["equity"] = df_trades["pnl_usd"].cumsum()

# Estadísticas
total = len(df_trades)
ganadores = df_trades[df_trades["pnl"] > 0]
perdedores = df_trades[df_trades["pnl"] < 0]
win_rate = len(ganadores) / total * 100
pnl_total = df_trades["pnl_usd"].sum()
ganancia_media = ganadores["pnl_usd"].mean()
perdida_media = perdedores["pnl_usd"].mean()
profit_factor = ganadores["pnl_usd"].sum() / abs(perdedores["pnl_usd"].sum())
max_drawdown = (df_trades["equity"].cummax() - df_trades["equity"]).max()

por_resultado = df_trades["resultado"].value_counts()

print("=" * 40)
print("ESTADÍSTICAS ORB SHORT — Mar/Abr 2026")
print("=" * 40)
print(f"Total trades:        {total}")
print(f"Ganadores:           {len(ganadores)} ({win_rate:.1f}%)")
print(f"Perdedores:          {len(perdedores)} ({100-win_rate:.1f}%)")
print(f"Por target:          {por_resultado.get('target', 0)}")
print(f"Por stop:            {por_resultado.get('stop', 0)}")
print(f"Por tiempo:          {por_resultado.get('tiempo', 0)}")
print(f"PnL total:           ${pnl_total:,.0f}")
print(f"Ganancia media:      ${ganancia_media:,.0f}")
print(f"Pérdida media:       ${perdida_media:,.0f}")
print(f"Profit factor:       {profit_factor:.2f}")
print(f"Max drawdown:        ${max_drawdown:,.0f}")
print("=" * 40)

# Curva de equity
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(1, total+1)),
    y=df_trades["equity"],
    mode="lines+markers",
    line=dict(color="royalblue", width=2),
    marker=dict(
        color=["green" if p > 0 else "red" for p in df_trades["pnl"]],
        size=8
    ),
    name="Equity"
))
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Curva de equity — ORB SHORT",
    xaxis_title="Trade #",
    yaxis_title="PnL acumulado (USD)",
    height=400,
    template="plotly_white"
)
fig.show()

ESTADÍSTICAS ORB SHORT — Mar/Abr 2026
Total trades:        13
Ganadores:           5 (38.5%)
Perdedores:          8 (61.5%)
Por target:          5
Por stop:            8
Por tiempo:          0
PnL total:           $-838
Ganancia media:      $930
Pérdida media:       $-686
Profit factor:       0.85
Max drawdown:        $3,250


In [8]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import glob
import os

# Leer todos los archivos de la carpeta datos
ruta = r"C:\Users\Usuario\Documents\mdc\TradingBacktest\datos\*.txt"
archivos = sorted(glob.glob(ruta))

print("Archivos encontrados:")
for a in archivos:
    print(f"  {os.path.basename(a)}")

# Leer y unir todos los contratos
dfs = []
for archivo in archivos:
    df_temp = pd.read_csv(
        archivo,
        sep=";",
        header=None,
        names=["datetime", "open", "high", "low", "close", "volume"]
    )
    df_temp["contrato"] = os.path.basename(archivo).replace(".txt", "")
    dfs.append(df_temp)

df_raw = pd.concat(dfs, ignore_index=True)
df_raw["datetime"] = pd.to_datetime(df_raw["datetime"], format="%Y%m%d %H%M%S")
df_raw = df_raw.sort_values("datetime").drop_duplicates(subset="datetime")
df_raw = df_raw.set_index("datetime")

print(f"\nTotal barras unidas: {len(df_raw)}")
print(f"Desde: {df_raw.index[0]}")
print(f"Hasta: {df_raw.index[-1]}")

Archivos encontrados:
  ES_DIC25.txt
  ES_JUN25.txt
  ES_JUN26.txt
  ES_MAR26.txt
  ES_SEP25.txt

Total barras unidas: 380052
Desde: 2025-03-16 19:29:00
Hasta: 2026-04-17 12:55:00


In [9]:
import pytz

eastern = pytz.timezone("America/New_York")
df_raw.index = df_raw.index.tz_localize("America/New_York")

# Filtrar período LONG: 25 mayo 2025 al 5 marzo 2026
start_long = "2025-05-25"
end_long = "2026-03-05"
df_period = df_raw[start_long:end_long]

# Filtrar horario sesión regular 9:30 a 16:00 ET
df_session = df_period.between_time("09:30", "16:00")

# Construir velas de 5 minutos
df_5m = df_session.resample("5min").agg({
    "open": "first",
    "high": "max",
    "low": "min",
    "close": "last",
    "volume": "sum"
}).dropna()

print(f"Total barras 5 minutos período LONG: {len(df_5m)}")
print(f"Desde: {df_5m.index[0]}")
print(f"Hasta: {df_5m.index[-1]}")
print(f"Días de trading: {df_5m.index.normalize().nunique()}")

Total barras 5 minutos período LONG: 15930
Desde: 2025-05-25 11:35:00-04:00
Hasta: 2026-03-05 16:00:00-05:00
Días de trading: 213


In [10]:
trades_long = []
dias = df_5m.groupby(df_5m.index.date)

for fecha, dia in dias:
    barras = dia.copy()
    if len(barras) < 6:
        continue
    
    # Verificar que el día arranca a las 9:30
    primera_hora = barras.index[0].time()
    if primera_hora > pd.Timestamp("09:35").time():
        continue
    
    # Rango ORB primeras 3 velas (9:30, 9:35, 9:40)
    orb = barras.iloc[:3]
    orb_high = orb["high"].max()
    orb_low = orb["low"].min()
    orb_rango = orb_high - orb_low
    
    # Filtro volatilidad
    if orb_rango > 20:
        continue
    
    # Buscar entrada LONG desde vela 4 en adelante
    entrada = None
    for i in range(3, len(barras)):
        vela = barras.iloc[i]
        if vela["close"] > orb_high:
            entrada = {
                "fecha": fecha,
                "hora_entrada": barras.index[i],
                "precio_entrada": vela["close"],
                "orb_high": orb_high,
                "orb_low": orb_low,
                "orb_rango": orb_rango,
                "stop": orb_low,
                "target": vela["close"] + (orb_rango * 1.5)
            }
            break
    
    if entrada is None:
        continue
    
    # Simular resultado
    resultado = None
    for i in range(barras.index.get_loc(entrada["hora_entrada"]) + 1, len(barras)):
        vela = barras.iloc[i]
        hora = barras.index[i].time()
        
        if hora >= pd.Timestamp("15:45").time():
            entrada["salida"] = vela["close"]
            entrada["hora_salida"] = barras.index[i]
            entrada["resultado"] = "tiempo"
            entrada["pnl"] = vela["close"] - entrada["precio_entrada"]
            resultado = entrada
            break
        
        if vela["low"] <= entrada["stop"]:
            entrada["salida"] = entrada["stop"]
            entrada["hora_salida"] = barras.index[i]
            entrada["resultado"] = "stop"
            entrada["pnl"] = entrada["stop"] - entrada["precio_entrada"]
            resultado = entrada
            break
        
        if vela["high"] >= entrada["target"]:
            entrada["salida"] = entrada["target"]
            entrada["hora_salida"] = barras.index[i]
            entrada["resultado"] = "target"
            entrada["pnl"] = entrada["target"] - entrada["precio_entrada"]
            resultado = entrada
            break
    
    if resultado:
        trades_long.append(resultado)

df_long = pd.DataFrame(trades_long)
print(f"Total trades LONG: {len(df_long)}")
print(df_long[["fecha","precio_entrada","orb_rango","stop","target","resultado","pnl"]].to_string())

Total trades LONG: 178
          fecha  precio_entrada  orb_rango     stop    target resultado     pnl
0    2025-05-26         5887.00       4.00  5882.25  5893.000      stop  -4.750
1    2025-05-27         5908.75       8.25  5899.75  5921.125      stop  -9.000
2    2025-05-28         5924.00       4.50  5918.50  5930.750    target   6.750
3    2025-05-30         5920.75       4.75  5914.25  5927.875      stop  -6.500
4    2025-06-02         5886.75       4.25  5880.75  5893.125    target   6.375
5    2025-06-03         5926.50       6.50  5918.50  5936.250    target   9.750
6    2025-06-04         5995.75       3.00  5991.25  6000.250      stop  -4.500
7    2025-06-05         5988.25       3.25  5984.00  5993.125      stop  -4.250
8    2025-06-06         5985.00       5.00  5969.25  5992.500    target   7.500
9    2025-06-09         6014.75       1.50  6012.75  6017.000      stop  -2.000
10   2025-06-10         6015.00       2.75  6011.25  6019.125      stop  -3.750
11   2025-06-11  

In [11]:
punto = 50

df_long["pnl_usd"] = df_long["pnl"] * punto
df_long["equity"] = df_long["pnl_usd"].cumsum()

# Estadísticas
total = len(df_long)
ganadores = df_long[df_long["pnl"] > 0]
perdedores = df_long[df_long["pnl"] < 0]
win_rate = len(ganadores) / total * 100
pnl_total = df_long["pnl_usd"].sum()
ganancia_media = ganadores["pnl_usd"].mean()
perdida_media = perdedores["pnl_usd"].mean()
profit_factor = ganadores["pnl_usd"].sum() / abs(perdedores["pnl_usd"].sum())
max_drawdown = (df_long["equity"].cummax() - df_long["equity"]).max()
por_resultado = df_long["resultado"].value_counts()

print("=" * 40)
print("ESTADÍSTICAS ORB LONG — May 2025 / Mar 2026")
print("=" * 40)
print(f"Total trades:        {total}")
print(f"Ganadores:           {len(ganadores)} ({win_rate:.1f}%)")
print(f"Perdedores:          {len(perdedores)} ({100-win_rate:.1f}%)")
print(f"Por target:          {por_resultado.get('target', 0)}")
print(f"Por stop:            {por_resultado.get('stop', 0)}")
print(f"Por tiempo:          {por_resultado.get('tiempo', 0)}")
print(f"PnL total:           ${pnl_total:,.0f}")
print(f"Ganancia media:      ${ganancia_media:,.0f}")
print(f"Pérdida media:       ${perdida_media:,.0f}")
print(f"Profit factor:       {profit_factor:.2f}")
print(f"Max drawdown:        ${max_drawdown:,.0f}")
print("=" * 40)

# Curva de equity
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(1, total+1)),
    y=df_long["equity"],
    mode="lines+markers",
    line=dict(color="royalblue", width=2),
    marker=dict(
        color=["green" if p > 0 else "red" for p in df_long["pnl"]],
        size=6
    ),
    name="Equity"
))
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Curva de equity — ORB LONG (May 2025 / Mar 2026)",
    xaxis_title="Trade #",
    yaxis_title="PnL acumulado (USD)",
    height=400,
    template="plotly_white"
)
fig.show()

ESTADÍSTICAS ORB LONG — May 2025 / Mar 2026
Total trades:        178
Ganadores:           79 (44.4%)
Perdedores:          99 (55.6%)
Por target:          78
Por stop:            98
Por tiempo:          2
PnL total:           $-1,319
Ganancia media:      $370
Pérdida media:       $-309
Profit factor:       0.96
Max drawdown:        $4,994


In [12]:
# Calcular cierre del día anterior
cierres_diarios = df_5m.groupby(df_5m.index.date)["close"].last()

trades_long_filtrado = []
dias = df_5m.groupby(df_5m.index.date)

for fecha, dia in dias:
    barras = dia.copy()
    if len(barras) < 6:
        continue
    
    # Verificar que el día arranca a las 9:30
    primera_hora = barras.index[0].time()
    if primera_hora > pd.Timestamp("09:35").time():
        continue
    
    # Obtener cierre del día anterior
    fechas_disponibles = [f for f in cierres_diarios.index if f < fecha]
    if not fechas_disponibles:
        continue
    cierre_ayer = cierres_diarios[fechas_disponibles[-1]]
    
    # Rango ORB primeras 3 velas
    orb = barras.iloc[:3]
    orb_high = orb["high"].max()
    orb_low = orb["low"].min()
    orb_rango = orb_high - orb_low
    
    # Filtro volatilidad
    if orb_rango > 20:
        continue
    
    # Filtro: precio de apertura por encima del cierre de ayer
    precio_apertura = barras.iloc[0]["open"]
    if precio_apertura <= cierre_ayer:
        continue
    
    # Buscar entrada LONG desde vela 4
    entrada = None
    for i in range(3, len(barras)):
        vela = barras.iloc[i]
        if vela["close"] > orb_high:
            entrada = {
                "fecha": fecha,
                "hora_entrada": barras.index[i],
                "precio_entrada": vela["close"],
                "orb_high": orb_high,
                "orb_low": orb_low,
                "orb_rango": orb_rango,
                "stop": orb_low,
                "target": vela["close"] + (orb_rango * 1.5),
                "cierre_ayer": cierre_ayer
            }
            break
    
    if entrada is None:
        continue
    
    # Simular resultado
    resultado = None
    for i in range(barras.index.get_loc(entrada["hora_entrada"]) + 1, len(barras)):
        vela = barras.iloc[i]
        hora = barras.index[i].time()
        
        if hora >= pd.Timestamp("15:45").time():
            entrada["salida"] = vela["close"]
            entrada["hora_salida"] = barras.index[i]
            entrada["resultado"] = "tiempo"
            entrada["pnl"] = vela["close"] - entrada["precio_entrada"]
            resultado = entrada
            break
        
        if vela["low"] <= entrada["stop"]:
            entrada["salida"] = entrada["stop"]
            entrada["hora_salida"] = barras.index[i]
            entrada["resultado"] = "stop"
            entrada["pnl"] = entrada["stop"] - entrada["precio_entrada"]
            resultado = entrada
            break
        
        if vela["high"] >= entrada["target"]:
            entrada["salida"] = entrada["target"]
            entrada["hora_salida"] = barras.index[i]
            entrada["resultado"] = "target"
            entrada["pnl"] = entrada["target"] - entrada["precio_entrada"]
            resultado = entrada
            break
    
    if resultado:
        trades_long_filtrado.append(resultado)

df_filtrado = pd.DataFrame(trades_long_filtrado)

# Estadísticas
punto = 50
df_filtrado["pnl_usd"] = df_filtrado["pnl"] * punto
df_filtrado["equity"] = df_filtrado["pnl_usd"].cumsum()

total = len(df_filtrado)
ganadores = df_filtrado[df_filtrado["pnl"] > 0]
perdedores = df_filtrado[df_filtrado["pnl"] < 0]
win_rate = len(ganadores) / total * 100
pnl_total = df_filtrado["pnl_usd"].sum()
ganancia_media = ganadores["pnl_usd"].mean()
perdida_media = perdedores["pnl_usd"].mean()
profit_factor = ganadores["pnl_usd"].sum() / abs(perdedores["pnl_usd"].sum())
max_drawdown = (df_filtrado["equity"].cummax() - df_filtrado["equity"]).max()
por_resultado = df_filtrado["resultado"].value_counts()

print("=" * 45)
print("ORB LONG + FILTRO CIERRE ANTERIOR — May 2025 / Mar 2026")
print("=" * 45)
print(f"Total trades:        {total}")
print(f"Ganadores:           {len(ganadores)} ({win_rate:.1f}%)")
print(f"Perdedores:          {len(perdedores)} ({100-win_rate:.1f}%)")
print(f"Por target:          {por_resultado.get('target', 0)}")
print(f"Por stop:            {por_resultado.get('stop', 0)}")
print(f"Por tiempo:          {por_resultado.get('tiempo', 0)}")
print(f"PnL total:           ${pnl_total:,.0f}")
print(f"Ganancia media:      ${ganancia_media:,.0f}")
print(f"Pérdida media:       ${perdida_media:,.0f}")
print(f"Profit factor:       {profit_factor:.2f}")
print(f"Max drawdown:        ${max_drawdown:,.0f}")
print("=" * 45)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(1, total+1)),
    y=df_filtrado["equity"],
    mode="lines+markers",
    line=dict(color="royalblue", width=2),
    marker=dict(
        color=["green" if p > 0 else "red" for p in df_filtrado["pnl"]],
        size=6
    ),
    name="Equity"
))
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Curva de equity — ORB LONG + filtro cierre anterior",
    xaxis_title="Trade #",
    yaxis_title="PnL acumulado (USD)",
    height=400,
    template="plotly_white"
)
fig.show()

ORB LONG + FILTRO CIERRE ANTERIOR — May 2025 / Mar 2026
Total trades:        110
Ganadores:           51 (46.4%)
Perdedores:          59 (53.6%)
Por target:          50
Por stop:            58
Por tiempo:          2
PnL total:           $2,525
Ganancia media:      $359
Pérdida media:       $-267
Profit factor:       1.16
Max drawdown:        $3,850


In [14]:
# Calcular VWAP desde las 18:00 ET de cada día
# Usamos df_raw que tiene todos los datos incluyendo pre-market

df_vwap = df_raw.copy()
df_vwap.index = df_vwap.index.tz_convert("America/New_York")

# Precio típico
df_vwap["typical_price"] = (df_vwap["high"] + df_vwap["low"] + df_vwap["close"]) / 3
df_vwap["tp_vol"] = df_vwap["typical_price"] * df_vwap["volume"]

# Definir sesión: cada sesión arranca a las 18:00 ET
def get_session_id(dt):
    if dt.hour >= 18:
        return dt.date()
    else:
        import datetime
        return (dt - datetime.timedelta(days=1)).date()

df_vwap["session_id"] = df_vwap.index.map(get_session_id)

# Calcular VWAP acumulado por sesión
df_vwap["cum_tp_vol"] = df_vwap.groupby("session_id")["tp_vol"].cumsum()
df_vwap["cum_vol"] = df_vwap.groupby("session_id")["volume"].cumsum()
df_vwap["vwap"] = df_vwap["cum_tp_vol"] / df_vwap["cum_vol"]

print("VWAP calculado correctamente")
print(df_vwap[["close", "volume", "vwap"]].head(10))

VWAP calculado correctamente
                             close  volume         vwap
datetime                                               
2025-03-16 19:29:00-04:00  5681.00       1  5681.000000
2025-03-16 22:01:00-04:00  5682.50     558  5680.417710
2025-03-16 22:02:00-04:00  5680.25     207  5680.349869
2025-03-16 22:03:00-04:00  5678.00      70  5680.215909
2025-03-16 22:04:00-04:00  5679.50      40  5680.160388
2025-03-16 22:05:00-04:00  5678.50      63  5680.043397
2025-03-16 22:06:00-04:00  5679.50      29  5680.017132
2025-03-16 22:07:00-04:00  5676.00     114  5679.681685
2025-03-16 22:08:00-04:00  5673.25     158  5678.915255
2025-03-16 22:09:00-04:00  5670.75     171  5677.986298


In [15]:
# Filtrar período LONG
df_vwap_period = df_vwap["2025-05-25":"2026-03-05"]

# Resamplear a 5 minutos
df_5m_vwap = df_vwap_period.resample("5min").agg({
    "open": "first",
    "high": "max",
    "low": "min",
    "close": "last",
    "volume": "sum",
    "vwap": "last",
    "session_id": "last"
}).dropna()

# Filtrar solo horario de sesión regular 9:30 a 16:00
df_5m_vwap = df_5m_vwap.between_time("09:30", "16:00")

print(f"Total barras: {len(df_5m_vwap)}")
print(df_5m_vwap[["open","high","low","close","vwap"]].head(10))

Total barras: 15931
                              open     high      low    close         vwap
datetime                                                                  
2025-05-25 11:35:00-04:00  5814.75  5814.75  5814.75  5814.75  5814.750000
2025-05-25 14:40:00-04:00  5814.75  5814.75  5814.75  5814.75  5814.750000
2025-05-26 09:30:00-04:00  5886.25  5886.25  5884.25  5884.50  5870.007417
2025-05-26 09:35:00-04:00  5884.50  5884.50  5882.50  5883.00  5870.026484
2025-05-26 09:40:00-04:00  5883.00  5883.75  5882.25  5883.50  5870.056594
2025-05-26 09:45:00-04:00  5883.50  5884.25  5883.50  5883.75  5870.069632
2025-05-26 09:50:00-04:00  5883.75  5884.75  5883.00  5884.00  5870.108192
2025-05-26 09:55:00-04:00  5883.75  5885.00  5883.50  5884.50  5870.153617
2025-05-26 10:00:00-04:00  5884.50  5885.00  5882.25  5885.00  5870.216505
2025-05-26 10:05:00-04:00  5884.75  5887.50  5884.75  5887.00  5870.269661


In [16]:
trades_vwap = []
dias = df_5m_vwap.groupby(df_5m_vwap.index.date)

for fecha, dia in dias:
    barras = dia.copy()
    if len(barras) < 4:
        continue

    for i in range(1, len(barras) - 1):
        vela = barras.iloc[i]
        vela_anterior = barras.iloc[i - 1]
        
        vwap_actual = vela["vwap"]
        vwap_anterior = vela_anterior["vwap"]
        
        if pd.isna(vwap_actual) or pd.isna(vwap_anterior):
            continue

        # VWAP alcista
        vwap_alcista = vwap_actual > vwap_anterior
        # VWAP bajista
        vwap_bajista = vwap_actual < vwap_anterior

        # Señal LONG: mínimo toca o cruza el VWAP y cierra por encima
        senal_long = (
            vwap_alcista and
            vela["low"] <= vwap_actual and
            vela["close"] > vwap_actual
        )

        # Señal SHORT: máximo toca o cruza el VWAP y cierra por debajo
        senal_short = (
            vwap_bajista and
            vela["high"] >= vwap_actual and
            vela["close"] < vwap_actual
        )

        if not senal_long and not senal_short:
            continue

        direccion = "LONG" if senal_long else "SHORT"
        precio_entrada = vela["close"]

        if direccion == "LONG":
            stop = vela["low"] - 0.25
            riesgo = precio_entrada - stop
            target = precio_entrada + (riesgo * 2)
        else:
            stop = vela["high"] + 0.25
            riesgo = stop - precio_entrada
            target = precio_entrada - (riesgo * 2)

        if riesgo <= 0 or riesgo > 20:
            continue

        # Simular resultado
        resultado = None
        for j in range(i + 1, len(barras)):
            vela_j = barras.iloc[j]
            hora = barras.index[j].time()

            if hora >= pd.Timestamp("15:45").time():
                resultado = {
                    "fecha": fecha,
                    "hora_entrada": barras.index[i],
                    "direccion": direccion,
                    "precio_entrada": precio_entrada,
                    "stop": stop,
                    "target": target,
                    "riesgo": riesgo,
                    "salida": vela_j["close"],
                    "hora_salida": barras.index[j],
                    "resultado": "tiempo",
                    "pnl": vela_j["close"] - precio_entrada if direccion == "LONG" else precio_entrada - vela_j["close"]
                }
                break

            if direccion == "LONG":
                if vela_j["low"] <= stop:
                    resultado = {"fecha": fecha, "hora_entrada": barras.index[i], "direccion": direccion,
                                "precio_entrada": precio_entrada, "stop": stop, "target": target,
                                "riesgo": riesgo, "salida": stop, "hora_salida": barras.index[j],
                                "resultado": "stop", "pnl": stop - precio_entrada}
                    break
                if vela_j["high"] >= target:
                    resultado = {"fecha": fecha, "hora_entrada": barras.index[i], "direccion": direccion,
                                "precio_entrada": precio_entrada, "stop": stop, "target": target,
                                "riesgo": riesgo, "salida": target, "hora_salida": barras.index[j],
                                "resultado": "target", "pnl": target - precio_entrada}
                    break
            else:
                if vela_j["high"] >= stop:
                    resultado = {"fecha": fecha, "hora_entrada": barras.index[i], "direccion": direccion,
                                "precio_entrada": precio_entrada, "stop": stop, "target": target,
                                "riesgo": riesgo, "salida": stop, "hora_salida": barras.index[j],
                                "resultado": "stop", "pnl": stop - precio_entrada}
                    break
                if vela_j["low"] <= target:
                    resultado = {"fecha": fecha, "hora_entrada": barras.index[i], "direccion": direccion,
                                "precio_entrada": precio_entrada, "stop": stop, "target": target,
                                "riesgo": riesgo, "salida": target, "hora_salida": barras.index[j],
                                "resultado": "target", "pnl": precio_entrada - target}
                    break

        if resultado:
            trades_vwap.append(resultado)

df_vwap_trades = pd.DataFrame(trades_vwap)
print(f"Total trades VWAP: {len(df_vwap_trades)}")
print(df_vwap_trades[["fecha","hora_entrada","direccion","precio_entrada","riesgo","resultado","pnl"]].head(20).to_string())

Total trades VWAP: 1209
         fecha              hora_entrada direccion  precio_entrada  riesgo resultado    pnl
0   2025-05-27 2025-05-27 12:45:00-04:00      LONG         5894.25    6.25      stop  -6.25
1   2025-05-27 2025-05-27 12:50:00-04:00     SHORT         5888.00    6.75      stop   6.75
2   2025-05-27 2025-05-27 12:55:00-04:00     SHORT         5891.50    0.50      stop   0.50
3   2025-05-27 2025-05-27 13:00:00-04:00      LONG         5893.25    2.75    target   5.50
4   2025-05-27 2025-05-27 13:30:00-04:00     SHORT         5879.75   12.00      stop  12.00
5   2025-05-27 2025-05-27 13:55:00-04:00     SHORT         5886.75    1.75      stop   1.75
6   2025-05-27 2025-05-27 14:00:00-04:00      LONG         5893.00    7.75    target  15.50
7   2025-05-28 2025-05-28 10:30:00-04:00      LONG         5934.00    5.25    target  10.50
8   2025-05-28 2025-05-28 13:50:00-04:00      LONG         5935.25    2.00      stop  -2.00
9   2025-05-28 2025-05-28 13:55:00-04:00     SHORT      

In [17]:
def calcular_pnl(row):
    if row["direccion"] == "LONG":
        return row["salida"] - row["precio_entrada"]
    else:
        return row["precio_entrada"] - row["salida"]

df_vwap_trades["pnl"] = df_vwap_trades.apply(calcular_pnl, axis=1)

print("PnL corregido. Primeros 20 trades:")
print(df_vwap_trades[["fecha","hora_entrada","direccion","precio_entrada","stop","salida","resultado","pnl"]].head(20).to_string())

PnL corregido. Primeros 20 trades:
         fecha              hora_entrada direccion  precio_entrada     stop   salida resultado    pnl
0   2025-05-27 2025-05-27 12:45:00-04:00      LONG         5894.25  5888.00  5888.00      stop  -6.25
1   2025-05-27 2025-05-27 12:50:00-04:00     SHORT         5888.00  5894.75  5894.75      stop  -6.75
2   2025-05-27 2025-05-27 12:55:00-04:00     SHORT         5891.50  5892.00  5892.00      stop  -0.50
3   2025-05-27 2025-05-27 13:00:00-04:00      LONG         5893.25  5890.50  5898.75    target   5.50
4   2025-05-27 2025-05-27 13:30:00-04:00     SHORT         5879.75  5891.75  5891.75      stop -12.00
5   2025-05-27 2025-05-27 13:55:00-04:00     SHORT         5886.75  5888.50  5888.50      stop  -1.75
6   2025-05-27 2025-05-27 14:00:00-04:00      LONG         5893.00  5885.25  5908.50    target  15.50
7   2025-05-28 2025-05-28 10:30:00-04:00      LONG         5934.00  5928.75  5944.50    target  10.50
8   2025-05-28 2025-05-28 13:50:00-04:00      L

In [18]:
punto = 50

df_vwap_trades["pnl_usd"] = df_vwap_trades["pnl"] * punto
df_vwap_trades["equity"] = df_vwap_trades["pnl_usd"].cumsum()

total = len(df_vwap_trades)
ganadores = df_vwap_trades[df_vwap_trades["pnl"] > 0]
perdedores = df_vwap_trades[df_vwap_trades["pnl"] < 0]
win_rate = len(ganadores) / total * 100
pnl_total = df_vwap_trades["pnl_usd"].sum()
ganancia_media = ganadores["pnl_usd"].mean()
perdida_media = perdedores["pnl_usd"].mean()
profit_factor = ganadores["pnl_usd"].sum() / abs(perdedores["pnl_usd"].sum())
max_drawdown = (df_vwap_trades["equity"].cummax() - df_vwap_trades["equity"]).max()
por_resultado = df_vwap_trades["resultado"].value_counts()
por_direccion = df_vwap_trades["direccion"].value_counts()

print("=" * 45)
print("ESTADÍSTICAS VWAP BOUNCE — May 2025 / Mar 2026")
print("=" * 45)
print(f"Total trades:        {total}")
print(f"Ganadores:           {len(ganadores)} ({win_rate:.1f}%)")
print(f"Perdedores:          {len(perdedores)} ({100-win_rate:.1f}%)")
print(f"Por target:          {por_resultado.get('target', 0)}")
print(f"Por stop:            {por_resultado.get('stop', 0)}")
print(f"Por tiempo:          {por_resultado.get('tiempo', 0)}")
print(f"LONG:                {por_direccion.get('LONG', 0)}")
print(f"SHORT:               {por_direccion.get('SHORT', 0)}")
print(f"PnL total:           ${pnl_total:,.0f}")
print(f"Ganancia media:      ${ganancia_media:,.0f}")
print(f"Pérdida media:       ${perdida_media:,.0f}")
print(f"Profit factor:       {profit_factor:.2f}")
print(f"Max drawdown:        ${max_drawdown:,.0f}")
print("=" * 45)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(1, total+1)),
    y=df_vwap_trades["equity"],
    mode="lines",
    line=dict(color="royalblue", width=1.5),
    name="Equity"
))
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Curva de equity — VWAP Bounce (May 2025 / Mar 2026)",
    xaxis_title="Trade #",
    yaxis_title="PnL acumulado (USD)",
    height=400,
    template="plotly_white"
)
fig.show()

ESTADÍSTICAS VWAP BOUNCE — May 2025 / Mar 2026
Total trades:        1209
Ganadores:           416 (34.4%)
Perdedores:          791 (65.6%)
Por target:          339
Por stop:            744
Por tiempo:          126
LONG:                622
SHORT:               587
PnL total:           $-14,300
Ganancia media:      $360
Pérdida media:       $-208
Profit factor:       0.91
Max drawdown:        $26,750


In [19]:
trades_vwap_v2 = []
dias = df_5m_vwap.groupby(df_5m_vwap.index.date)

for fecha, dia in dias:
    barras = dia.copy()
    if len(barras) < 6:
        continue

    for i in range(3, len(barras) - 1):
        vela = barras.iloc[i]
        vela_anterior = barras.iloc[i - 1]

        vwap_actual = vela["vwap"]
        vwap_anterior = vela_anterior["vwap"]

        if pd.isna(vwap_actual) or pd.isna(vwap_anterior):
            continue

        vwap_alcista = vwap_actual > vwap_anterior
        vwap_bajista = vwap_actual < vwap_anterior

        # Filtro 3 velas previas por encima del VWAP para LONG
        tres_velas_long = all(
            barras.iloc[i - k]["close"] > barras.iloc[i - k]["vwap"]
            for k in range(1, 4)
        )

        # Filtro 3 velas previas por debajo del VWAP para SHORT
        tres_velas_short = all(
            barras.iloc[i - k]["close"] < barras.iloc[i - k]["vwap"]
            for k in range(1, 4)
        )

        senal_long = (
            vwap_alcista and
            tres_velas_long and
            vela["low"] <= vwap_actual and
            vela["close"] > vwap_actual
        )

        senal_short = (
            vwap_bajista and
            tres_velas_short and
            vela["high"] >= vwap_actual and
            vela["close"] < vwap_actual
        )

        if not senal_long and not senal_short:
            continue

        direccion = "LONG" if senal_long else "SHORT"
        precio_entrada = vela["close"]

        if direccion == "LONG":
            stop = vela["low"] - 0.25
            riesgo = precio_entrada - stop
            target = precio_entrada + (riesgo * 2)
        else:
            stop = vela["high"] + 0.25
            riesgo = stop - precio_entrada
            target = precio_entrada - (riesgo * 2)

        if riesgo <= 0 or riesgo > 20:
            continue

        resultado = None
        for j in range(i + 1, len(barras)):
            vela_j = barras.iloc[j]
            hora = barras.index[j].time()

            if hora >= pd.Timestamp("15:45").time():
                resultado = {
                    "fecha": fecha,
                    "hora_entrada": barras.index[i],
                    "direccion": direccion,
                    "precio_entrada": precio_entrada,
                    "stop": stop,
                    "target": target,
                    "riesgo": riesgo,
                    "salida": vela_j["close"],
                    "hora_salida": barras.index[j],
                    "resultado": "tiempo",
                    "pnl": vela_j["close"] - precio_entrada if direccion == "LONG" else precio_entrada - vela_j["close"]
                }
                break

            if direccion == "LONG":
                if vela_j["low"] <= stop:
                    resultado = {"fecha": fecha, "hora_entrada": barras.index[i], "direccion": direccion,
                                "precio_entrada": precio_entrada, "stop": stop, "target": target,
                                "riesgo": riesgo, "salida": stop, "hora_salida": barras.index[j],
                                "resultado": "stop", "pnl": stop - precio_entrada}
                    break
                if vela_j["high"] >= target:
                    resultado = {"fecha": fecha, "hora_entrada": barras.index[i], "direccion": direccion,
                                "precio_entrada": precio_entrada, "stop": stop, "target": target,
                                "riesgo": riesgo, "salida": target, "hora_salida": barras.index[j],
                                "resultado": "target", "pnl": target - precio_entrada}
                    break
            else:
                if vela_j["high"] >= stop:
                    resultado = {"fecha": fecha, "hora_entrada": barras.index[i], "direccion": direccion,
                                "precio_entrada": precio_entrada, "stop": stop, "target": target,
                                "riesgo": riesgo, "salida": stop, "hora_salida": barras.index[j],
                                "resultado": "stop", "pnl": stop - precio_entrada}
                    break
                if vela_j["low"] <= target:
                    resultado = {"fecha": fecha, "hora_entrada": barras.index[i], "direccion": direccion,
                                "precio_entrada": precio_entrada, "stop": stop, "target": target,
                                "riesgo": riesgo, "salida": target, "hora_salida": barras.index[j],
                                "resultado": "target", "pnl": precio_entrada - target}
                    break

        if resultado:
            trades_vwap_v2.append(resultado)

df_vwap_v2 = pd.DataFrame(trades_vwap_v2)

# Corregir pnl
def calcular_pnl(row):
    if row["direccion"] == "LONG":
        return row["salida"] - row["precio_entrada"]
    else:
        return row["precio_entrada"] - row["salida"]

df_vwap_v2["pnl"] = df_vwap_v2.apply(calcular_pnl, axis=1)

print(f"Total trades: {len(df_vwap_v2)}")
print(df_vwap_v2[["fecha","hora_entrada","direccion","precio_entrada","riesgo","resultado","pnl"]].head(20).to_string())

Total trades: 425
         fecha              hora_entrada direccion  precio_entrada  riesgo resultado    pnl
0   2025-05-27 2025-05-27 13:55:00-04:00     SHORT         5886.75    1.75      stop  -1.75
1   2025-05-28 2025-05-28 13:50:00-04:00      LONG         5935.25    2.00      stop  -2.00
2   2025-05-29 2025-05-29 13:30:00-04:00      LONG         5943.75    6.75      stop  -6.75
3   2025-05-30 2025-05-30 10:25:00-04:00     SHORT         5914.25    1.25      stop  -1.25
4   2025-05-30 2025-05-30 10:45:00-04:00      LONG         5915.00    0.50    target   1.00
5   2025-05-30 2025-05-30 10:50:00-04:00      LONG         5915.00    0.50      stop  -0.50
6   2025-05-30 2025-05-30 13:30:00-04:00     SHORT         5899.25   13.75      stop -13.75
7   2025-05-30 2025-05-30 14:05:00-04:00     SHORT         5908.00    2.75      stop  -2.75
8   2025-05-30 2025-05-30 14:45:00-04:00      LONG         5913.75    5.50      stop  -5.50
9   2025-06-02 2025-06-02 12:25:00-04:00      LONG         589

In [20]:
punto = 50

df_vwap_v2["pnl_usd"] = df_vwap_v2["pnl"] * punto
df_vwap_v2["equity"] = df_vwap_v2["pnl_usd"].cumsum()

total = len(df_vwap_v2)
ganadores = df_vwap_v2[df_vwap_v2["pnl"] > 0]
perdedores = df_vwap_v2[df_vwap_v2["pnl"] < 0]
win_rate = len(ganadores) / total * 100
pnl_total = df_vwap_v2["pnl_usd"].sum()
ganancia_media = ganadores["pnl_usd"].mean()
perdida_media = perdedores["pnl_usd"].mean()
profit_factor = ganadores["pnl_usd"].sum() / abs(perdedores["pnl_usd"].sum())
max_drawdown = (df_vwap_v2["equity"].cummax() - df_vwap_v2["equity"]).max()
por_resultado = df_vwap_v2["resultado"].value_counts()
por_direccion = df_vwap_v2["direccion"].value_counts()

print("=" * 50)
print("VWAP BOUNCE v2 (3 velas) — May 2025 / Mar 2026")
print("=" * 50)
print(f"Total trades:        {total}")
print(f"Ganadores:           {len(ganadores)} ({win_rate:.1f}%)")
print(f"Perdedores:          {len(perdedores)} ({100-win_rate:.1f}%)")
print(f"Por target:          {por_resultado.get('target', 0)}")
print(f"Por stop:            {por_resultado.get('stop', 0)}")
print(f"Por tiempo:          {por_resultado.get('tiempo', 0)}")
print(f"LONG:                {por_direccion.get('LONG', 0)}")
print(f"SHORT:               {por_direccion.get('SHORT', 0)}")
print(f"PnL total:           ${pnl_total:,.0f}")
print(f"Ganancia media:      ${ganancia_media:,.0f}")
print(f"Pérdida media:       ${perdida_media:,.0f}")
print(f"Profit factor:       {profit_factor:.2f}")
print(f"Max drawdown:        ${max_drawdown:,.0f}")
print("=" * 50)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(1, total+1)),
    y=df_vwap_v2["equity"],
    mode="lines",
    line=dict(color="royalblue", width=1.5),
    name="Equity"
))
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Curva de equity — VWAP Bounce v2 (May 2025 / Mar 2026)",
    xaxis_title="Trade #",
    yaxis_title="PnL acumulado (USD)",
    height=400,
    template="plotly_white"
)
fig.show()


VWAP BOUNCE v2 (3 velas) — May 2025 / Mar 2026
Total trades:        425
Ganadores:           149 (35.1%)
Perdedores:          276 (64.9%)
Por target:          129
Por stop:            261
Por tiempo:          35
LONG:                234
SHORT:               191
PnL total:           $2,438
Ganancia media:      $334
Pérdida media:       $-172
Profit factor:       1.05
Max drawdown:        $6,788


In [21]:
# Calcular ATR de 14 períodos sobre las velas de 5 minutos
df_5m_vwap["prev_close"] = df_5m_vwap["close"].shift(1)
df_5m_vwap["tr"] = df_5m_vwap[["high", "low", "prev_close"]].apply(
    lambda row: max(
        row["high"] - row["low"],
        abs(row["high"] - row["prev_close"]),
        abs(row["low"] - row["prev_close"])
    ), axis=1
)
df_5m_vwap["atr"] = df_5m_vwap["tr"].rolling(14).mean()

print("ATR calculado correctamente")
print(df_5m_vwap[["close", "atr"]].dropna().head(10))

ATR calculado correctamente
                             close       atr
datetime                                    
2025-05-26 10:25:00-04:00  5887.75  6.642857
2025-05-26 10:30:00-04:00  5887.50  6.767857
2025-05-26 10:35:00-04:00  5886.50  6.875000
2025-05-26 10:40:00-04:00  5887.00  1.875000
2025-05-26 10:45:00-04:00  5888.50  1.857143
2025-05-26 10:50:00-04:00  5887.25  1.892857
2025-05-26 10:55:00-04:00  5885.25  2.053571
2025-05-26 11:00:00-04:00  5885.00  2.000000
2025-05-26 11:05:00-04:00  5885.50  1.982143
2025-05-26 11:10:00-04:00  5887.50  1.964286


In [22]:
trades_vwap_v3 = []
dias = df_5m_vwap.groupby(df_5m_vwap.index.date)

for fecha, dia in dias:
    barras = dia.copy()
    if len(barras) < 6:
        continue

    for i in range(3, len(barras) - 1):
        vela = barras.iloc[i]
        vela_anterior = barras.iloc[i - 1]

        vwap_actual = vela["vwap"]
        vwap_anterior = vela_anterior["vwap"]
        atr_actual = vela["atr"]

        if pd.isna(vwap_actual) or pd.isna(vwap_anterior) or pd.isna(atr_actual):
            continue

        # Stop basado en 25% del ATR
        stop_distancia = round(atr_actual * 0.25, 2)

        # Mínimo 2 puntos, máximo 8 puntos
        if stop_distancia < 2 or stop_distancia > 8:
            continue

        vwap_alcista = vwap_actual > vwap_anterior
        vwap_bajista = vwap_actual < vwap_anterior

        # Filtro 3 velas previas
        tres_velas_long = all(
            barras.iloc[i - k]["close"] > barras.iloc[i - k]["vwap"]
            for k in range(1, 4)
        )
        tres_velas_short = all(
            barras.iloc[i - k]["close"] < barras.iloc[i - k]["vwap"]
            for k in range(1, 4)
        )

        senal_long = (
            vwap_alcista and
            tres_velas_long and
            vela["low"] <= vwap_actual and
            vela["close"] > vwap_actual
        )

        senal_short = (
            vwap_bajista and
            tres_velas_short and
            vela["high"] >= vwap_actual and
            vela["close"] < vwap_actual
        )

        if not senal_long and not senal_short:
            continue

        direccion = "LONG" if senal_long else "SHORT"
        precio_entrada = vela["close"]

        if direccion == "LONG":
            stop = precio_entrada - stop_distancia
            target = precio_entrada + (stop_distancia * 2)
        else:
            stop = precio_entrada + stop_distancia
            target = precio_entrada - (stop_distancia * 2)

        resultado = None
        for j in range(i + 1, len(barras)):
            vela_j = barras.iloc[j]
            hora = barras.index[j].time()

            if hora >= pd.Timestamp("15:45").time():
                resultado = {
                    "fecha": fecha,
                    "hora_entrada": barras.index[i],
                    "direccion": direccion,
                    "precio_entrada": precio_entrada,
                    "stop": stop,
                    "target": target,
                    "stop_distancia": stop_distancia,
                    "atr": atr_actual,
                    "salida": vela_j["close"],
                    "hora_salida": barras.index[j],
                    "resultado": "tiempo",
                    "pnl": vela_j["close"] - precio_entrada if direccion == "LONG" else precio_entrada - vela_j["close"]
                }
                break

            if direccion == "LONG":
                if vela_j["low"] <= stop:
                    resultado = {"fecha": fecha, "hora_entrada": barras.index[i], "direccion": direccion,
                                "precio_entrada": precio_entrada, "stop": stop, "target": target,
                                "stop_distancia": stop_distancia, "atr": atr_actual,
                                "salida": stop, "hora_salida": barras.index[j],
                                "resultado": "stop", "pnl": stop - precio_entrada}
                    break
                if vela_j["high"] >= target:
                    resultado = {"fecha": fecha, "hora_entrada": barras.index[i], "direccion": direccion,
                                "precio_entrada": precio_entrada, "stop": stop, "target": target,
                                "stop_distancia": stop_distancia, "atr": atr_actual,
                                "salida": target, "hora_salida": barras.index[j],
                                "resultado": "target", "pnl": target - precio_entrada}
                    break
            else:
                if vela_j["high"] >= stop:
                    resultado = {"fecha": fecha, "hora_entrada": barras.index[i], "direccion": direccion,
                                "precio_entrada": precio_entrada, "stop": stop, "target": target,
                                "stop_distancia": stop_distancia, "atr": atr_actual,
                                "salida": stop, "hora_salida": barras.index[j],
                                "resultado": "stop", "pnl": stop - precio_entrada}
                    break
                if vela_j["low"] <= target:
                    resultado = {"fecha": fecha, "hora_entrada": barras.index[i], "direccion": direccion,
                                "precio_entrada": precio_entrada, "stop": stop, "target": target,
                                "stop_distancia": stop_distancia, "atr": atr_actual,
                                "salida": target, "hora_salida": barras.index[j],
                                "resultado": "target", "pnl": precio_entrada - target}
                    break

        if resultado:
            trades_vwap_v3.append(resultado)

df_vwap_v3 = pd.DataFrame(trades_vwap_v3)

def calcular_pnl(row):
    if row["direccion"] == "LONG":
        return row["salida"] - row["precio_entrada"]
    else:
        return row["precio_entrada"] - row["salida"]

df_vwap_v3["pnl"] = df_vwap_v3.apply(calcular_pnl, axis=1)

print(f"Total trades: {len(df_vwap_v3)}")
print(df_vwap_v3[["fecha","hora_entrada","direccion","precio_entrada","atr","stop_distancia","resultado","pnl"]].head(20).to_string())

Total trades: 70
         fecha              hora_entrada direccion  precio_entrada        atr  stop_distancia resultado   pnl
0   2025-05-30 2025-05-30 13:30:00-04:00     SHORT         5899.25   8.000000            2.00      stop -2.00
1   2025-05-30 2025-05-30 14:05:00-04:00     SHORT         5908.00   9.089286            2.27      stop -2.27
2   2025-05-30 2025-05-30 14:45:00-04:00      LONG         5913.75   8.196429            2.05      stop -2.05
3   2025-06-02 2025-06-02 14:15:00-04:00     SHORT         5892.50  10.410714            2.60      stop -2.60
4   2025-07-16 2025-07-16 15:55:00-04:00     SHORT         6281.00  14.982143            3.75    tiempo  2.50
5   2025-08-12 2025-08-12 14:00:00-04:00      LONG         6415.50   8.678571            2.17    target  4.34
6   2025-08-21 2025-08-21 14:05:00-04:00     SHORT         6402.25   8.410714            2.10      stop -2.10
7   2025-09-02 2025-09-02 13:55:00-04:00     SHORT         6412.50   8.821429            2.21      stop